In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
from pyspark.sql import Row

import datetime
users = [
    {
        "id": 1,
        "first_name": "Corrie",
        "last_name": "Van den Oord",
        "email": "cvandenoord0@etsy.com",
        "phone_numbers": Row(mobile="+1 234 567 8901", home="+1 234 567 8911"),
        "courses": [1, 2],
        "is_customer": True,
        "amount_paid": 1000.55,
        "customer_from": datetime.date(2021, 1, 15),
        "last_updated_ts": datetime.datetime(2021, 2, 10, 1, 15, 0)
    },
    {
        "id": 2,
        "first_name": "Nikolaus",
        "last_name": "Brewitt",
        "email": "nbrewitt1@dailymail.co.uk",
        "phone_numbers":  Row(mobile="+1 234 567 8923", home="1 234 567 8934"),
        "courses": [3],
        "is_customer": True,
        "amount_paid": 900.0,
        "customer_from": datetime.date(2021, 2, 14),
        "last_updated_ts": datetime.datetime(2021, 2, 18, 3, 33, 0)
    },
    {
        "id": 3,
        "first_name": "Orelie",
        "last_name": "Penney",
        "email": "openney2@vistaprint.com",
        "phone_numbers": Row(mobile="+1 714 512 9752", home="+1 714 512 6601"),
        "courses": [2, 4],
        "is_customer": True,
        "amount_paid": 850.55,
        "customer_from": datetime.date(2021, 1, 21),
        "last_updated_ts": datetime.datetime(2021, 3, 15, 15, 16, 55)
    },
    {
        "id": 4,
        "first_name": "Ashby",
        "last_name": "Maddocks",
        "email": "amaddocks3@home.pl",
        "phone_numbers": Row(mobile=None, home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 10, 17, 45, 30)
    },
    {
        "id": 5,
        "first_name": "Kurt",
        "last_name": "Rome",
        "email": "krome4@shutterfly.com",
        "phone_numbers": Row(mobile="+1 817 934 7142", home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 2, 0, 55, 18)
    }
]

In [ ]:
import pandas as pd

df_users = spark.createDataFrame(pd.DataFrame(users))
df_users.show()

In [ ]:
help(df_users.sort)

In [ ]:
#Сортируем в возрастающем порядке по колонке first_name
df_users.sort('first_name').show()

In [ ]:
#Сортируем в возрастающем порядке по колонке first_name
#Помним, что к колонке можно обратиться разными способами
df_users.sort(df_users.first_name).show()

In [ ]:
#Помним, что к колонке можно обратиться разными способами

from pyspark.sql.functions import col

df_users.sort(col('first_name')).show()

In [ ]:
#Пример с использованием функции, добавлении новой колонки и сортировки по новой колонке
from pyspark.sql.functions import size

df_users. \
    select('id', 'first_name', 'courses'). \
    withColumn('no_of_courses', size('courses')). \
    sort(size('courses')).show()

In [ ]:
#Пример с сортировкой в обратном порядке
df_users.sort('first_name', ascending=False).show()

In [ ]:
#Аналогичный результат можно получить с использованием функции desc
from pyspark.sql.functions import desc

df_users.sort(desc('first_name')).show()

In [ ]:
#Пример с использованием функции, добавлении новой колонки и сортировки по новой колонке

df_users. \
    select('id', 'courses'). \
    withColumn('no_of_courses', size('courses')). \
    sort(desc('no_of_courses')). \
    show()

In [ ]:
#Сортируем с учетом null и желаемого поведения null при сортировке
df_users. \
    select('id', 'customer_from'). \
    sort(col('customer_from').asc_nulls_last()). \
    show()

In [ ]:
#Пример с orderBy. sort и orderBy взаимозаменяемые функции 
df_users. \
    select('id', 'customer_from'). \
    orderBy(df_users['customer_from'].asc_nulls_first()). \
    show()

In [ ]:
spark.stop()